# Stage 0: Setup

In [ ]:
### Imports
### System
import sys, os

### I/O
import h5py
import tables_io

### Operations
import numpy as np

### RAIL
from rail.core.data import DataStore
from rail.core.stage import RailStage
from rail.core.data import PqHandle

sys.path.insert(0, "/global/homes/s/sajkov/rail_umap/src/degrading")
from MultiSurveyErrorModel import MultiSurveyErrorModel

os.environ["LEPHAREDIR"] = "/pscratch/sd/s/sajkov/lephare/data"
os.environ["LEPHAREWORK"] = "/pscratch/sd/s/sajkov/lephare/work"
from rail.estimation.algos.lephare import LephareInformer, LephareEstimator
import lephare as lp

sys.path.insert(0, "/global/homes/s/sajkov/rail_umap/src/estimation")
from UMAPEstimator import UMAPEstimator

### Set the date, start the timer
import time
date = time.strftime('%d%b%y', time.localtime())

LEPHAREDIR is being set to the default cache directory:
/global/homes/s/sajkov/.cache/lephare/data
More than 1Gb may be written there.
LEPHAREWORK is being set to the default cache directory:
/global/homes/s/sajkov/.cache/lephare/work
Default work cache is already linked. 
This is linked to the run directory:
/global/homes/s/sajkov/.cache/lephare/runs/20260702T160835


In [ ]:
### for testing only
import matplotlib.pyplot as plt
plt.style.use("/global/homes/s/sajkov/umap_nz_cal.mplstyle")

In [ ]:
### Initialize random state
seed = 42
rng = np.random.default_rng(seed = seed)

In [ ]:
### Specify outputs directory
outputs_directory = f"/pscratch/sd/s/sajkov/analysis_pipeline/runs/{date}"
if not os.path.exists(outputs_directory):
    os.makedirs(outputs_directory)

# Stage 1: Create datasets


In [ ]:
### Specify path to noiseless catalog and redshifts
noiseless_catalog_filepath = "/pscratch/sd/s/sajkov/data/integrated_catalog_23apr26.pq"
redshifts_filepath = "/pscratch/sd/s/sajkov/data/mock_catalog_Ch1_26.h5"

In [ ]:
DATASET_popCosmos_full   = tables_io.read(noiseless_catalog_filepath)
REDSHIFTS_popCosmos_full = h5py.File(redshifts_filepath)['sps_parameters'][:, -1]

In [ ]:
### Number of pop-cosmos sources to use in analysis
data_cut = 100_000

### Fraction of deep field versus WFD photometry
deep_field_frac = 0.2

### Randomize full dataset indices, take `deep_field_frac` to be the deep field, let the rest be WFD
RANDIDX_popCosmos_full = rng.choice(np.arange(len(DATASET_popCosmos_full)), len(DATASET_popCosmos_full), replace = False)
RANDIDX_popCosmos_cut  = RANDIDX_popCosmos_full[:int(data_cut * (1 + deep_field_frac))]

deep_field_cut = int(deep_field_frac * data_cut)
IDX_popCosmos_DeepField             = RANDIDX_popCosmos_cut[:deep_field_cut]
IDX_popCosmos_DeepField_lpReference = RANDIDX_popCosmos_cut[deep_field_cut:2*deep_field_cut]
IDX_popCosmos_WideFastDeep          = RANDIDX_popCosmos_cut[2*deep_field_cut:]

In [ ]:
### 5-sigma limiting depths ------------------
### LSST: median values for COSMOS deep field from https://usdf-maf.slac.stanford.edu/summaryStats?runId=5#Basics_Coadd%20M5
### Roman from https://github.com/jfcrenshaw/photerr/blob/a014b39729ddde3daf80be2dbe82f5a7f958882c/photerr/roman.py#L120-L126
### HSC Niji (60 min exposure) from https://sites.google.com/view/hsc-mb-survey3/filter-specification?authuser=0 
###
### Acessed June 3, 2026
### --------------------------------------------

M5_DEPTHS_DeepField = {'LSST_u'    : 27.74,
                      'LSST_g'    : 28.69,
                      'LSST_r'    : 28.88,
                      'LSST_i'    : 28.96,
                      'LSST_z'    : 28.26,
                      'LSST_y'    : 26.63,
                      'Roman_F062': 27.7,
                      'Roman_F087': 27.7,
                      'Roman_F106': 27.6,
                      'Roman_F129': 27.5,
                      'Roman_F158': 27.0,
                      'Roman_F184': 25.9,
                      'Roman_F213': 28.3,
                      'HSC_MB_00' : 26.41,
                      'HSC_MB_01' : 26.51,
                      'HSC_MB_02' : 26.45,
                      'HSC_MB_03' : 26.69,
                      'HSC_MB_04' : 26.93,
                      'HSC_MB_05' : 26.62,
                      'HSC_MB_06' : 26.26,
                      'HSC_MB_07' : 26.02,
                      'HSC_MB_08' : 26.07,
                      'HSC_MB_09' : 26.00,
                      'HSC_MB_10' : 26.06,
                      'HSC_MB_11' : 25.52,
                      'HSC_MB_12' : 25.58,
                      'HSC_MB_13' : 25.43,
                      'HSC_MB_14' : 25.15,
                      'HSC_MB_15' : 24.79}


### 5-sigma limiting depths for WideFastDeep from https://usdf-maf.slac.stanford.edu/summaryStats?runId=5#Basics_Coadd%20M5
### Column `DD:WFD CoaddM5`
M5_DEPTHS_WideFastDeep = {'LSST_u'    : 25.61,
                          'LSST_g'    : 26.90,
                          'LSST_r'    : 26.87,
                          'LSST_i'    : 26.43,
                          'LSST_z'    : 25.73,
                          'LSST_y'    : 24.79}

### Get list of bands
BANDS_DeepField = list(M5_DEPTHS_DeepField.keys())
BANDS_WideFastDeep = list(M5_DEPTHS_WideFastDeep.keys())

In [ ]:
### Select needed bands and pick out needed sources
PHOTOMETRY_DeepField_noiseless                = DATASET_popCosmos_full[BANDS_DeepField].iloc[IDX_popCosmos_DeepField]
PHOTOMETRY_DeepField_lpReference_noiseless    = DATASET_popCosmos_full[BANDS_DeepField].iloc[IDX_popCosmos_DeepField_lpReference]
PHOTOMETRY_WideFastDeep_noiseless             = DATASET_popCosmos_full[BANDS_WideFastDeep].iloc[IDX_popCosmos_WideFastDeep]

### Same as above, for redshifts
REDSHIFTS_DeepField                = REDSHIFTS_popCosmos_full[IDX_popCosmos_DeepField]
REDSHIFTS_DeepField_lpReference    = REDSHIFTS_popCosmos_full[IDX_popCosmos_DeepField_lpReference]
REDSHIFTS_WideFastDeep             = REDSHIFTS_popCosmos_full[IDX_popCosmos_WideFastDeep]

## Apply noise

In [ ]:
### Noising parameters

nYrObs     = 1 # one-year depths
nVisYr     = 1 # one visit/yr (i.e., no co-adds)
gamma      = 0.04
sigLim     = 0 # 
inputType  = 'pogson' # input pogson magnitudes (AB)
outputType = 'asinh'  # output asinh magnitudes

seed = 42

### DP 1.1: Deep, multi-band, medium-band photometry + degraded LSST photometry

In [ ]:
noisyPhotometryPath_DeepField = f"{outputs_directory}/PHOTOMETRY_DeepField_noisy_{date}.pq"

getNoisyDeepFieldPhotometry = MultiSurveyErrorModel.make_stage(
    name = "getNoisyDeepFieldPhotometry",
    
    inputType  = inputType,
    outputType = outputType,

    noisy_catalog     = noisyPhotometryPath_DeepField,
    
    m5     = M5_DEPTHS_DeepField,
    bands  = BANDS_DeepField,
    nYrObs = nYrObs,
    nVisYr = nVisYr,
    gamma  = gamma,
    sigLim = sigLim,
    
    seed = seed
)

getNoisyDeepFieldPhotometry.set_data("noiseless_catalog", PHOTOMETRY_DeepField_noiseless) 
getNoisyDeepFieldPhotometry.run()
getNoisyDeepFieldPhotometry.get_handle("noisy_catalog").write()
getNoisyDeepFieldPhotometry.finalize()

In [ ]:
noisyPhotometryPath_DeepField_LePhareReference = f"{outputs_directory}/PHOTOMETRY_DeepField_noisy_lpReference_{date}.pq"

getNoisyDeepFieldPhotometry_lpReference = MultiSurveyErrorModel.make_stage(
    name = "getNoisyDeepFieldPhotometry_lpReference",
    
    inputType  = inputType,
    outputType = outputType,

    noisy_catalog     = noisyPhotometryPath_DeepField_LePhareReference,
    
    m5     = M5_DEPTHS_DeepField,
    bands  = BANDS_DeepField,
    nYrObs = nYrObs,
    nVisYr = nVisYr,
    gamma  = gamma,
    sigLim = sigLim,
    
    seed = seed
)

getNoisyDeepFieldPhotometry_lpReference.set_data("noiseless_catalog", PHOTOMETRY_DeepField_lpReference_noiseless) 
getNoisyDeepFieldPhotometry_lpReference.run()
getNoisyDeepFieldPhotometry_lpReference.get_handle("noisy_catalog").write()
getNoisyDeepFieldPhotometry_lpReference.finalize()

### *(((RETURN TO THIS)))* DP 1.2: same as DP 1.1, but with WideFastDeep 5 sigma depths

In [ ]:
# degradedPhotometryPath_DeepField = f"{outputs_directory}/PHOTOMETRY_DeepField_degraded_{date}.pq"

# degradeDeepFieldPhotometry = MultiSurveyErrorModel.make_stage(
#     name = "degradeDeepFieldPhotometry",
    
#     inputType  = inputType,
#     outputType = outputType,

#     noisy_catalog     = degradedPhotometryPath_DeepField,
    
#     m5     = M5_DEPTHS_WideFastDeep,
#     bands  = BANDS_WideFastDeep,
#     nYrObs = nYrObs,
#     nVisYr = nVisYr,
#     gamma  = gamma,
#     sigLim = sigLim,
    
#     seed = seed
# )

# degradeDeepFieldPhotometry.set_data("noiseless_catalog", PHOTOMETRY_DeepField_noiseless) 
# degradeDeepFieldPhotometry.run()
# degradeDeepFieldPhotometry.get_handle("noisy_catalog").write()
# degradeDeepFieldPhotometry.finalize()

### DP 1.3: LSST photometry with no spec-zs

In [ ]:
noisyPhotometryPath_WideFastDeep = f"{outputs_directory}//PHOTOMETRY_WideFastDeep_noisy_{date}.pq"

getNoisyWideFastDeepPhotometry = MultiSurveyErrorModel.make_stage(
    name = "getNoisyWideFastDeepPhotometry",
    
    inputType  = inputType,
    outputType = outputType,

    noisy_catalog     = noisyPhotometryPath_WideFastDeep,
    
    m5     = M5_DEPTHS_WideFastDeep,
    bands  = BANDS_WideFastDeep,
    nYrObs = nYrObs,
    nVisYr = nVisYr,
    gamma  = gamma,
    sigLim = sigLim,
    
    seed = seed
)

getNoisyWideFastDeepPhotometry.set_data("noiseless_catalog", PHOTOMETRY_WideFastDeep_noiseless) 
getNoisyWideFastDeepPhotometry.run()
getNoisyWideFastDeepPhotometry.get_handle("noisy_catalog").write()
getNoisyWideFastDeepPhotometry.finalize()

### Check outputs

In [ ]:
PHOTOMETRY_DeepField_noisy    = tables_io.read(f"{outputs_directory}/PHOTOMETRY_DeepField_noisy_03Jul26.pq")
PHOTOMETRY_DeepField_lpRefernce_noisy    = tables_io.read(f"{outputs_directory}/PHOTOMETRY_DeepField_noisy_03Jul26.pq")
PHOTOMETRY_WideFastDeep_noisy = tables_io.read(f"{outputs_directory}/PHOTOMETRY_WideFastDeep_noisy_03Jul26.pq")

# Stage 2: etimate photo-zs with LePhare on DP 1.1

In [ ]:
from rail.utils.path_utils import RAILDIR

In [ ]:
trainFile = os.path.join(RAILDIR, 'rail/examples_data/testdata/output_table_conv_train.hdf5')
testFile = os.path.join(RAILDIR, 'rail/examples_data/testdata/output_table_conv_test.hdf5')
# traindata_io = tables_io.read(trainFile)
# testdata_io = tables_io.read(testFile)

In [ ]:
lephare_config_file = os.path.join(RAILDIR, 'rail/examples_data/estimation_data/data/lsst.para')

In [ ]:
import shutil
shutil.copy(lephare_config_file, "/pscratch/sd/s/sajkov/analysis_pipeline")

In [ ]:
traindata_io.keys()

In [ ]:
testdata_io.keys()

In [ ]:
lephare_config

In [ ]:
lephare_config_file = os.path.join(RAILDIR, 'rail/examples_data/estimation_data/data/lsst.para')
lephare_config = lp.read_config(lephare_config_file)

lp.data_retrieval.get_auxiliary_data(keymap=lephare_config)

# Stage 3: inform two UMAPs:

### DP 3.1: with photoz-s

### DP 3.2: with spec-zs

# Stage 4: get photo-zs for DP 1.2

### DP 4.1: photo-zs from 3.1

### DP 4.2: photo-zs from 3.2

# Stage 5: compare photo-zs